# 00 · Préparer son poste et les données

Tout le TP travaille sur **un même fichier** : les courses des taxis jaunes de New York en mars 2026 (près de 4 millions de lignes). Ce notebook vérifie votre poste et télécharge les données, que les notebooks suivants convertiront dans tous les formats du cours.

## 1. Vérifier les bibliothèques

On affiche la version de Python (il faut 3.11 ou plus), puis celle de chaque bibliothèque du TP.

In [1]:
import platform
from importlib.metadata import version, PackageNotFoundError

print("Python", platform.python_version())
bibliotheques = ["pandas", "pyarrow", "openpyxl", "python-calamine", "deltalake", "pyiceberg",
                 "charset-normalizer", "matplotlib", "requests",
                 "sqlalchemy", "psycopg", "pymongo", "python-dotenv"]
for nom in bibliotheques:
    try:
        print("OK", nom, version(nom))
    except PackageNotFoundError:
        print("MANQUANT", nom)

Python 3.12.13
OK pandas 3.0.6
OK pyarrow 26.0.0
OK openpyxl 3.1.5
OK python-calamine 0.8.3
OK deltalake 1.6.6
OK pyiceberg 0.12.0
OK charset-normalizer 3.5.2
OK matplotlib 3.11.2
OK requests 2.34.2
OK sqlalchemy 2.1.4
OK psycopg 3.3.6
OK pymongo 4.18.3
OK python-dotenv 1.2.4


Ici, tout est `OK`. Une ligne `MANQUANT` ? Lancez `uv sync` (ou `pip install -r requirements.txt`) à la racine du projet, puis redémarrez le noyau.

## 2. Créer les dossiers

`data/` reçoit les fichiers téléchargés, `output/` tout ce que les notebooks produisent. Le notebook tourne dans `notebooks/`, d'où le `..` (dossier parent).

In [2]:
from pathlib import Path

DATA = Path("../data")
OUTPUT = Path("../output")
DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
print(DATA.exists(), OUTPUT.exists())

True True


`True True` : les deux dossiers existent (`exist_ok=True` évite une erreur s'ils étaient déjà là).

## 3. Télécharger les données

On télécharge deux fichiers dans `data/` : les courses (Parquet, 68 Mo) et la table des 265 zones de taxi (CSV). La fonction `telecharger` ne fait rien si le fichier est déjà là ; le premier téléchargement prend de quelques secondes à quelques minutes selon votre connexion.

In [3]:
import requests

def telecharger(url, chemin):
    """Télécharge url dans chemin, sauf si le fichier existe déjà."""
    if chemin.exists():
        print("Déjà présent :", chemin.name)
    else:
        reponse = requests.get(url, timeout=60)
        reponse.raise_for_status()
        chemin.write_bytes(reponse.content)
        print("Téléchargé :", chemin.name)

In [4]:
chemin_parquet = DATA / "yellow_tripdata_2026-03.parquet"
chemin_zones = DATA / "taxi_zone_lookup.csv"
telecharger("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-03.parquet", chemin_parquet)
telecharger("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv", chemin_zones)

Déjà présent : yellow_tripdata_2026-03.parquet
Déjà présent : taxi_zone_lookup.csv


Au premier lancement, la cellule affiche « Téléchargé », ensuite « Déjà présent » : rien n'est retéléchargé, on peut la relancer sans risque.

## 4. Premier coup d'œil, sans pandas

Un fichier n'est qu'une suite d'octets : le **format** dit comment les lire. Commençons par la taille.

In [5]:
print("Courses (Parquet) :", round(chemin_parquet.stat().st_size / 1_000_000, 1), "Mo")
print("Zones (CSV) :", round(chemin_zones.stat().st_size / 1_000, 1), "Ko")

Courses (Parquet) : 67.9 Mo
Zones (CSV) : 12.3 Ko


Près de 4 millions de courses tiennent dans 68 Mo : Parquet est compressé. Les 265 zones ne pèsent que 12 Ko.

Ouvrons le CSV comme un simple fichier texte et lisons ses 3 premières lignes.

In [6]:
with open(chemin_zones, encoding="utf-8") as fichier:
    for i in range(3):
        print(fichier.readline().strip())

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"


C'est lisible : une ligne d'en-tête avec les noms des colonnes, puis une zone par ligne, valeurs séparées par des virgules. Rien n'indique les types : pour le fichier, `1` n'est que du texte.

Même chose avec le Parquet : on lit ses 40 premiers octets en mode binaire (`"rb"`). C'est la question de la slide 4 : que voit-on en l'ouvrant dans le Bloc-notes ?

In [7]:
with open(chemin_parquet, "rb") as fichier:
    print(fichier.read(40))

b'PAR1\x15\x04\x15\x18\x15*L\x15\x06\x15\x00\x12\x00\x00(\xb5/\xfd \x0ca\x00\x00\x02\x00\x00\x00\x07\x00\x00\x00\x01\x00\x00\x00\x15'


Illisible : c'est du binaire, sauf les 4 premiers octets, `PAR1`, la signature du format Parquet. Pour lire le contenu, il faut un outil comme pandas (notebook 01).

## 5. Le fichier `.env` (notebook 07 seulement)

Le notebook 07 se connecte aux bases du TP « Sources de données » : il lui faut le fichier `.env` (les identifiants) à la racine du projet. On vérifie seulement qu'il existe, sans jamais afficher son contenu.

In [8]:
if Path("../.env").exists():
    print(".env présent")
else:
    print(".env absent : copiez le .env du TP Sources de données (mêmes identifiants)")

.env présent


S'il est absent, copiez-le à côté de `.env.example` avant le notebook 07 (les autres notebooks n'en ont pas besoin). Git l'ignore : il ne sera pas publié.

## À retenir

- Python 3.11 ou plus, et `OK` pour chaque bibliothèque.
- `data/` contient les courses de mars 2026 (Parquet, 68 Mo) et les zones (CSV, 12 Ko).
- Un CSV est du texte lisible, sans types ; un Parquet est du binaire compressé (signature `PAR1`).
- Le `.env` est en place avant le notebook 07.

Suite : notebook 01, les bases de pandas.